# Learning Rate Schedules and Warmup Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: Schedule Functions

Each function takes the current step and returns the learning rate at that step.

In [ ]:
```python

import math

def constant_schedule(step, lr=0.01, **kwargs):

    return lr

def step_decay_schedule(step, lr=0.1, step_size=100, gamma=0.1, **kwargs):

    return lr * (gamma ** (step // step_size))

def cosine_schedule(step, lr=0.01, total_steps=1000, lr_min=1e-5, **kwargs):

    if step >= total_steps:

        return lr_min

    return lr_min + 0.5 * (lr - lr_min) * (1 + math.cos(math.pi * step / total_steps))

def warmup_cosine_schedule(step, lr=0.01, total_steps=1000, warmup_steps=100, lr_min=1e-5, **kwargs):

    if total_steps <= warmup_steps:

        return lr * (step / max(warmup_steps, 1))

    if step < warmup_steps:

        return lr * step / warmup_steps

    progress = (step - warmup_steps) / (total_steps - warmup_steps)

    return lr_min + 0.5 * (lr - lr_min) * (1 + math.cos(math.pi * progress))

def one_cycle_schedule(step, lr=0.01, total_steps=1000, **kwargs):

    mid = max(total_steps // 2, 1)

    if step < mid:

        return (lr / 25) + (lr - lr / 25) * step / mid

    else:

        progress = (step - mid) / max(total_steps - mid, 1)

        return lr * (1 - progress) + (lr / 10000) * progress

In [ ]:
```

### Step 2: Visualize All Schedules

Print a text-based plot showing how each schedule evolves over training.

In [ ]:
```python

def visualize_schedule(name, schedule_fn, total_steps=500, **kwargs):

    steps = list(range(0, total_steps, total_steps // 20))

    if total_steps - 1 not in steps:

        steps.append(total_steps - 1)

    lrs = [schedule_fn(s, total_steps=total_steps, **kwargs) for s in steps]

    max_lr = max(lrs) if max(lrs) > 0 else 1.0

    print(f"\n{name}:")

    for s, lr_val in zip(steps, lrs):

        bar_len = int(lr_val / max_lr * 40)

        bar = "#" * bar_len

        print(f"  Step {s:4d}: lr={lr_val:.6f} {bar}")

In [ ]:
```

### Step 3: Training Network

A simple two-layer network on the circle dataset, same as previous lessons, but now we vary the schedule.

In [ ]:
```python

import random

def sigmoid(x):

    x = max(-500, min(500, x))

    return 1.0 / (1.0 + math.exp(-x))

def relu(x):

    return max(0.0, x)

def relu_deriv(x):

    return 1.0 if x > 0 else 0.0

def make_circle_data(n=200, seed=42):

    random.seed(seed)

    data = []

    for _ in range(n):

        x = random.uniform(-2, 2)

        y = random.uniform(-2, 2)

        label = 1.0 if x * x + y * y < 1.5 else 0.0

        data.append(([x, y], label))

    return data

def train_with_schedule(schedule_fn, schedule_name, data, epochs=300, base_lr=0.05, **kwargs):

    random.seed(0)

    hidden_size = 8

    total_steps = epochs * len(data)

    std = math.sqrt(2.0 / 2)

    w1 = [[random.gauss(0, std) for _ in range(2)] for _ in range(hidden_size)]

    b1 = [0.0] * hidden_size

    w2 = [random.gauss(0, std) for _ in range(hidden_size)]

    b2 = 0.0

    step = 0

    epoch_losses = []

    for epoch in range(epochs):

        total_loss = 0

        correct = 0

        for x, target in data:

            lr = schedule_fn(step, lr=base_lr, total_steps=total_steps, **kwargs)

            z1 = []

            h = []

            for i in range(hidden_size):

                z = w1[i][0] * x[0] + w1[i][1] * x[1] + b1[i]

                z1.append(z)

                h.append(relu(z))

            z2 = sum(w2[i] * h[i] for i in range(hidden_size)) + b2

            out = sigmoid(z2)

            error = out - target

            d_out = error * out * (1 - out)

            for i in range(hidden_size):

                d_h = d_out * w2[i] * relu_deriv(z1[i])

                w2[i] -= lr * d_out * h[i]

                for j in range(2):

                    w1[i][j] -= lr * d_h * x[j]

                b1[i] -= lr * d_h

            b2 -= lr * d_out

            total_loss += (out - target) ** 2

            if (out >= 0.5) == (target >= 0.5):

                correct += 1

            step += 1

        avg_loss = total_loss / len(data)

        accuracy = correct / len(data) * 100

        epoch_losses.append(avg_loss)

    return epoch_losses

In [ ]:
```

### Step 4: Compare All Schedules

Train the same network with each schedule and compare final loss and convergence behavior.

In [ ]:
```python

def compare_schedules(data):

    configs = [

        ("Constant", constant_schedule, {}),

        ("Step Decay", step_decay_schedule, {"step_size": 15000, "gamma": 0.1}),

        ("Cosine", cosine_schedule, {"lr_min": 1e-5}),

        ("Warmup+Cosine", warmup_cosine_schedule, {"warmup_steps": 3000, "lr_min": 1e-5}),

        ("1cycle", one_cycle_schedule, {}),

    ]

    print(f"\n{'Schedule':<20} {'Start Loss':>12} {'Mid Loss':>12} {'End Loss':>12} {'Best Loss':>12}")

    print("-" * 70)

    for name, schedule_fn, extra_kwargs in configs:

        losses = train_with_schedule(schedule_fn, name, data, epochs=300, base_lr=0.05, **extra_kwargs)

        mid_idx = len(losses) // 2

        best = min(losses)

        print(f"{name:<20} {losses[0]:>12.6f} {losses[mid_idx]:>12.6f} {losses[-1]:>12.6f} {best:>12.6f}")

In [ ]:
```

### Step 5: LR Too High vs Too Low

Demonstrate the three failure modes: too high (divergence), too low (crawling), and just right.

In [ ]:
```python

def lr_sensitivity(data):

    learning_rates = [1.0, 0.1, 0.01, 0.001, 0.0001]

    print("\nLR Sensitivity (constant schedule, 100 epochs):")

    print(f"  {'LR':>10} {'Start Loss':>12} {'End Loss':>12} {'Status':>15}")

    print("  " + "-" * 52)

    for lr in learning_rates:

        losses = train_with_schedule(constant_schedule, f"lr={lr}", data, epochs=100, base_lr=lr)

        start = losses[0]

        end = losses[-1]

        if end > start or math.isnan(end) or end > 1.0:

            status = "DIVERGED"

        elif end > start * 0.9:

            status = "BARELY MOVED"

        elif end < 0.15:

            status = "CONVERGED"

        else:

            status = "LEARNING"

        end_str = f"{end:.6f}" if not math.isnan(end) else "NaN"

        print(f"  {lr:>10.4f} {start:>12.6f} {end_str:>12} {status:>15}")

In [ ]:
```

## Exercises

In [ ]:
1. Implement exponential decay: lr(t) = lr_0 * gamma^t where gamma = 0.999. Compare to cosine annealing on the circle dataset.

2. Implement the learning rate range test (Leslie Smith): train for a few hundred steps while exponentially increasing the LR from 1e-7 to 1. Plot loss vs LR. The optimal max LR is just before the loss starts increasing.

3. Train with warmup + cosine but vary the warmup length: 0%, 1%, 5%, 10%, 20% of total steps. Find the sweet spot where training is most stable.

4. Implement cosine annealing with warm restarts (SGDR): reset the learning rate to lr_max every T steps and decay again. Compare to standard cosine on a longer training run.

5. Build a "schedule surgeon" that monitors training loss and automatically switches from warmup to cosine when the loss stabilizes, and reduces lr if the loss plateaus for too long.